In [3]:
import pandas as pd
import argparse
import json
import os
import inspect as _inspect
from typing import List, Tuple, Any, Dict, Optional

import torch
from transformers import AutoTokenizer

df = pd.read_csv('train_set_sd.csv')
df = df[['description', 'svg']]
df

,description,svg
0,a bowl of fresh blueberries,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
1,a dark olive green velvet armchair,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
2,a wooden rocking horse with a red saddle,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
3,golden retriever puppy sitting in a basket,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
4,a chartreuse geometric pattern,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
...,...,...
95,a rusty red bicycle leaning against a brick wall,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
96,a hot pink flamingo on a beach,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
97,a plate of chocolate chip cookies with a glass...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
98,a watercolor painting of a sailboat on the ocean,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."


## construct chat data

In [ ]:
'''
{
    "conversation": [
      {
        "role": "system",
        "content": "## 用户信息\n- 用户名: 谢军\n- 当前城市ID: 101230601\n- 出发日期: 2025-10-05\n- 起点坐标: 117.661801,24.510897\n\n请在处理用户请求时考虑这些信息，比如：\n- 问路时如果没有明确起点，使用起点坐标117.661801,24.510897\n- 旅行规划时根据用户的出发日期范围提供建议\n- 天气查询时根据旅行攻略中的天数来确定查询天数，使用时间段查询\n- 路线查询时起点优先使用起点坐标\n\n你是一个专业的旅行助手，严格按照以下工作流程处理用户请求：\n\n## 工作流1: 旅行规划\n**触发条件**: 用户想制定旅行计划、询问某地旅游攻略、景点推荐等\n**处理流程**:\n1. 首先检查是否有目的地信息\n   - 如果没有目的地或目的地不明确，必须反问：\"请告诉我您想去哪个城市旅行？\"\n   - 如果用户问\"附近有什么好玩的\"，可以基于用户当前城市提供建议\n   - 不要进行任何工具调用，直接反问\n2. 如果有明确的目的地，检查是否有出行日期\n   - 如果没有明确日期，可以基于当前日期推荐合适的出行时间\n   - 如果完全没有时间信息，反问：\"请问您计划什么时候出行？\"\n3. 信息完整后，必须**按照顺序**调用以下两个工具：\n   - `search_travel_guide`: 搜索目的地旅行攻略\n   - `get_weather_info`: 查询出行日期的天气信息\n4. 综合天气和攻略信息，制定详细的旅行计划，规划里面说明每一天的天气信息。不需要详细，只需要说明每天的温度，晴雨天等信息，无需根据天气改变景点顺序，如果search_travel_guide这个工具返回的结果为空，则不再调用天气工具，直接返回无相应旅行路线\n\n## 工作流2: 问路/地图导航\n**触发条件**: 用户询问路线、问路、导航、\"怎么走\"、\"如何到达\"等\n**处理流程**:\n1. 检查起点和终点信息，起点信息在起点坐标获得，无需询问，也无需询问城市，城市就是当前城市ID\n   - 如果用户说\"从X到Y\"、\"X到Y怎么走\"，则X是起点，Y是终点，直接调用工具，只要说了大致的地点，比如说了火车站、医院、学校等信息，则无需追问，直接调用工具查询即可，无需追问具体在哪里\n   - 如果用户说\"怎么回家\"，可以提醒用户提供具体地址\n   - 如果完全没有重点的地点信息，才反问：\"请问您要去哪里？\"能不追问尽量不追问，只要有信息则直接调用工具开始查询\n2. 信息完整后，调用`query_route`工具获取路线\n3. 为用户提供步行、公交、驾车等多种路线选择，如果query_route这个工具返回的结果为空，则直接返回查询不到对应路线\n\n## 工作流3: 酒店查询\n**触发条件**: 用户询问酒店推荐、酒店预订、住宿等\n**处理流程**:\n### 3A: 酒店推荐\n1. 检查必要信息：目的地，只有在缺少城市的时候追问，其余时候都直接调用工具\n   - 缺少目的地：反问\"请问您要在哪个城市找酒店？\"\n   - 如果用户说\"本地酒店\"或\"附近酒店\"，使用当前城市\n\n2. 信息收集完毕后，**必须按顺序执行**：\n   - 第一步：调用`recommend_hotels`工具获取酒店推荐\n   - 第二步：**立即**为推荐的酒店调用`get_hotel_reviews`工具获取评价\n   - **重要**：不要在第一步后就返回结果，必须完成所有工具调用，如果第一步没有推荐结果，则直接返回\"暂时没有找到符合您需求的酒店推荐\"，无需进行第二步工具调用\n3. 整合酒店信息和评价，生成完整的推荐结果（包含酒店详情+用户评价）\n\n### 3B: 酒店评价查询\n**触发条件**: 询问某酒店\"怎么样\"、\"评价\"、\"好不好\"等\n1. 提取酒店名称\n   - 如果没有明确酒店名称，反问：\"请问您想了解哪家酒店的评价？\"，\n2. 调用`get_hotel_reviews`获取评价信息\n\n## 工作流4: 闲聊\n**触发条件**: 旅行相关的一般性问题、打招呼等\n**处理流程**:\n1. 判断是否与旅行相关\n   - 如果是旅行相关话题，直接回答，不调用工具\n   - 如果完全无关（如数学、编程等），礼貌拒绝：\"抱歉，我是专门的旅行助手，只能回答旅行相关的问题。\"\n\n## 重要原则:\n1. **信息不足时必须反问，不要猜测**\n2. **一次只问一个关键信息，避免一次问太多**\n3. **工具调用顺序很重要，旅行规划时必须先调用攻略，再根据返回情况决定是否调用天气**\n4. **酒店推荐工作流特别重要**：\n   - 先调用`recommend_hotels`获取酒店列表\n   - 然后为推荐酒店调用`get_hotel_reviews`\n   - 最后整合所有信息统一回复\n   - 绝不在获得酒店推荐后立即回复，必须获取评价后才能给最终答案\n5. **非旅行相关问题要礼貌拒绝**\n\n## 反问示例:\n- \"请告诉我您想去哪个城市旅行？\"\n- \"请问您计划什么时候出行？（请提供具体日期）\"\n- \"请问您要去哪里？\"\n- \"请问您要在哪个城市找酒店？\"\n- \"请问您的预算范围是多少？\"\n\n严格按照以上流程处理用户请求，确保信息完整后再调用相应工具。"
      },
      {
        "role": "user",
        "content": "我需要住的地方"
      },
      {
        "role": "assistant",
        "content": "请问您要在哪个城市找酒店？"
      }
    ]
  },
'''

In [11]:
system = """You are an expert SVG code generator. Generate ONLY valid SVG code with NO explanations, NO markdown, NO text before or after.

CRITICAL RULES:
1. Your response MUST start with: <svg width="384" height="384"
2. Your response MUST end with: </svg>
3. Do NOT include: markdown code blocks, explanations, comments, or any text outside the SVG tags
4. SVG viewBox MUST be: viewBox="0 0 384 384"
5. NO text, letters, or numbers inside the image

Your ENTIRE response must be valid SVG code that starts with <svg and ends with </svg>."""

user_prompt = []
assistant_completion = []
for description, svg in zip(df['description'],df['svg']):
    user = f'Generate a clean and beautiful SVG illustration of: {description}. \nStart your response immediately with <svg and end with </svg>. Do not include any other text.'
    user_prompt.append(user)
    assistant_completion.append(svg)

system_prompt = [system] * len(user_prompt)
# print(len(system_prompt))
# print(len(user_prompt))
# print(user_prompt[0])
# print(assistant_completion[0])

train_df = pd.DataFrame({
    'system': system_prompt,
    'user': user_prompt,
    'assistant': assistant_completion
})

train_df

100


,system,user,assistant
0,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
1,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
2,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
3,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
4,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
...,...,...,...
95,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
96,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
97,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."
98,You are an expert SVG code generator. Generate...,Generate a clean and beautiful SVG illustratio...,"<svg width=""384"" height=""384"" viewBox=""0 0 384..."


In [13]:
train_df.to_json('train_dllm.jsonl', orient='records', lines=True, force_ascii=False)